# Node/Edgelists and Export
The primary aim of this book is to create a cleaned dataset, that can directly be used for further network and NLP analysis. The next step will therefore be to create node and edgelists of the data we so far have gathered and cleaned.

## Imports

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
import pickle
import importlib.util
import sqlite3
import itertools
from collections import defaultdict
import gc

# Load the module
spec = importlib.util.spec_from_file_location("4_utils.citation_utils", "4_utils/citation_utils.py")
citation_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(citation_utils)

# Assign functions to local variables
filter_nodes, process_db, save_to_csv = citation_utils.filter_nodes, citation_utils.process_db, citation_utils.save_to_csv

## Load Data

In [2]:
# Read resolved queried_publications and queried_references from JSON files
queried_publications = pd.read_json('2_data/queried_publications_lang_translated_openai_tokenized.json', lines=True)
queried_references = pd.read_json('2_data/queried_references_lang_translated_openai_tokenized.json', lines=True)

In [25]:
import pandas as pd

# Assuming queried_publications is your DataFrame
filtered_df = queried_publications[queried_publications['Author'].str.contains('Treder', case=False, na=False)]

# Save the filtered DataFrame to an Excel file
filtered_df.to_excel('filtered_publications.xlsx', index=False)

# Create Edgelists and other export formats for direct citations, co-citations, author-co-citations and bibliometric-coupling
Now we concatenate the exported queried publications and their exported references to build a basis for citation-analysis. For a direct citation network we need the citing sources and the cited references. Their relations will be saved as edges between their bibcodes in an edgelist. To match those edges to nodes and their attributes, we need a dataframe where the attributes  of all citing and cited nodes are combined.

In [3]:
# Ensure that the loaded data is in DataFrame format
if isinstance(queried_publications, list):
    queried_publications = pd.DataFrame(queried_publications)
if isinstance(queried_references, list):
    queried_references = pd.DataFrame(queried_references)

# Concatenate, sort, remove duplicates, and drop the temporary 'source' column
df_all_nodes = pd.concat([
    queried_publications.assign(source='queried_publications'), 
    queried_references.assign(source='queried_references')
]).sort_values(by='source').drop_duplicates(subset='Bibcode', keep="first").drop(columns='source')

# Rename the 'Bibcode' column to 'id' and show information about the dataframe
df_all_nodes.rename(columns={'Bibcode': 'id'}, inplace=True)

In [4]:
df_all_nodes

,id,Author,Title,Year,Journal,Journal Abbreviation,Issue,Volume,First Page,Last Page,...,Category,Citation Count,References,PDF_URL,Title_lang,Abstract_lang,Title_en,Abstract_en,full_text,tokens
0,1995PASP..107..803U,"Urry CM, Padovani P",Unified Schemes for Radio-Loud Active Galactic...,1995,Publications of the Astronomical Society of th...,PASP,,107,803,,...,article,4380.0,"[1966Natur.209..751H, 1966Natur.211..468R, 196...",https://ui.adsabs.harvard.edu/link_gateway/199...,en,en,Unified Schemes for Radio-Loud Active Galactic...,The appearance of active galactic nuclei (AGN)...,Unified Schemes for Radio-Loud Active Galactic...,"[unify, schemes, radio, loud, active, galactic..."
120518,1989Geobi..22..267B,"Bréhéret JG, Delamette M",Faunal fluctuations related to oceanographical...,1989,G&eacute;obios,Geobi,,22,267,277,...,article,3.0,"[1980Natur.286..252H, 1981Natur.293..291F, 198...",None,en,en,Faunal fluctuations related to oceanographical...,Planktonic foraminifera and ammonites containe...,Faunal fluctuations related to oceanographical...,"[faunal, fluctuation, relate, oceanographical,..."
120519,1990PhLB..237...52G,"Goto T, Okada Y",Wormhole solutions in scalar theories with non...,1990,Physics Letters B,PhLB,1,237,52,58,...,article,4.0,"[1987PhLB..195..337H, 1988NuPhB.310..643C, 198...",None,en,en,Wormhole solutions in scalar theories with non...,A path-integral formulation is given for a sys...,Wormhole solutions in scalar theories with non...,"[wormhole, solution, scalar, theory, non, abel..."
120520,1987ATJHT.109..717B,"Bayazitoglu Y, Lam TT",Marangoni convection in radiating fluids,1987,ASME Journal of Heat Transfer,ATJHT,,109,717,721,...,article,4.0,[],None,en,en,Marangoni convection in radiating fluids,The onset of Marangoni convection driven by su...,Marangoni convection in radiating fluids. The ...,"[marangoni, convection, radiating, fluid, onse..."
120521,1977NCimL..20..563F,Fabbri R,Ray aberration and large-scale anisotropy of t...,1977,Nuovo Cimento Lettere,NCimL,,20,563,568,...,article,3.0,[],None,en,en,Ray aberration and large-scale anisotropy of t...,Calculations are performed to show how the lar...,Ray aberration and large-scale anisotropy of t...,"[ray, aberration, large, scale, anisotropy, co..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169742,1981PhDT........86M,Miller GE,The Effects of Galaxy Collisions on the Struct...,1981,Ph.D. Thesis,PhDT,,,,,...,phdthesis,5.0,NaN,NaN,en,en,The Effects of Galaxy Collisions on the Struct...,The role of galaxy collisions in determining t...,The Effects of Galaxy Collisions on the Struct...,"[effects, galaxy, collisions, structure, evolu..."
169741,1986PhDT.........3J,Jung GV,The Hard X-Ray to Low-Energy Gamma-Ray Spectru...,1986,Ph.D. Thesis,PhDT,,,,,...,phdthesis,6.0,NaN,NaN,en,en,The Hard X-Ray to Low-Energy Gamma-Ray Spectru...,The spectrum of the Crab Nebula has been deter...,The Hard X-Ray to Low-Energy Gamma-Ray Spectru...,"[hard, ray, low, energy, gamma, ray, spectrum,..."
169740,1965PhRvL..14..398B,"Bose SK, Shirokov YM",Relativistic Extension of SU(6),1965,Physical Review Letters,PhRvL,11,14,398,399,...,article,7.0,NaN,NaN,en,en,Relativistic Extension of SU(6),,Relativistic Extension of SU(6).,"[relativistic, extension]"
169752,1969NW.....56...85B,Brosche P,Eine Analogie zwischen Elementarteilchen und a...,1969,Naturwissenschaften,NW,2,56,85,86,...,article,5.0,NaN,NaN,de,en,An analogy between elementary particles and as...,,An analogy between elementary particles and as...,"[analogy, elementary, particle, astronomical, ..."


In [ ]:
df_all_nodes.columns

We check again, if there are any duplicates in the dataframe.

In [5]:
# Check for any duplicated entries based on the 'id' column in the 'df_all_nodes' DataFrame.
duplicated_entries = df_all_nodes[df_all_nodes.duplicated(['id'], keep=False)].sort_values("id")

# If duplicated_entries DataFrame is not empty, print the first 10 entries, else print "No duplicated entries found."
if not duplicated_entries.empty:
    print("First 10 duplicated entries:")
    print(duplicated_entries.head(10))
else:
    print("No duplicated entries found.")

No duplicated entries found.


We now load the pickle files created above, to get the queried publications and their references.

In [6]:
# Load the variables 'bibcodes' and 'references' from the pickle file.
with open("2_data/bibcodes_references.pkl", "rb") as f:
    bibcodes, references, esources, fulltext_urls_dict = pickle.load(f)

And then create a DataFrame from our list of references, adding a 'source' column which consists of the Bibcodes of the citing publications.

In [7]:
#| column: page
#| label: df_all_biocodes

# Convert the list of references into a pandas DataFrame
df_all_bibcodes = pd.DataFrame(references)

# Insert a new column 'source' at index 0, populated with the bibcodes list
df_all_bibcodes.insert(0, "source", bibcodes)

# df_all_bibcodes.info()
df_all_bibcodes

,source,0,1,2,3,4,5,6,7,8,...,1439,1440,1441,1442,1443,1444,1445,1446,1447,1448
0,1986rgeg.book.....E,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,1999PhRvD..60j4002B,1979PhLB...82...60S,1979PhLB...84...83C,1980PhLB...90..249B,1980PhRvL..44..912M,1981PhLB..105..439I,1981PhRvD..24.1681D,1981ZPhyC..11..153S,1982NuPhB.196..475E,1982PhLB..115..193N,...,None,None,None,None,None,None,None,None,None,None
2,1998LRR.....1....1R,1957PhRv..108.1063R,1963ApJ...137..777R,1964ApJ...140..552J,1967ApJ...149..591T,1967ApJ...150.1005H,1967MNRAS.136..293L,1968ApJ...153..807H,1969ApJ...155..163P,1969ApJ...158..997T,...,None,None,None,None,None,None,None,None,None,None
3,1998JMP....39.3296B,1977JMP....18.2511P,1988CMaPh.117..353W,1988CQGra...5.1187B,1988CQGra...5.1543B,1988NuPhB.311...46W,1989CMaPh.123..177M,1989CMaPh.125..417H,1990IJMPA...5...93K,1991CQGra...8...41C,...,None,None,None,None,None,None,None,None,None,None
4,1996PhRvD..53..762H,1977PhRvD..16..245F,1978RSPSA.360..117B,1980PhLB...91...99S,1980PhRvD..22.1882B,1982qfcs.book.....B,1984NuPhB.244..541A,1986JETPL..44..619K,1987JETP...66..433K,1987PhLB..193..427M,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180780,1998BASI...26..577G,1990MNRAS.242..318S,1991Natur.351..719R,1994ApJ...437...67G,1995ApJ...454...69P,1995ApJ...455..108L,1996A&A...308L...5P,1996A&A...315L..32C,1996A&A...315L..64L,1996A&A...315L..71B,...,None,None,None,None,None,None,None,None,None,None
180781,1999ASPC..192..163R,1988ApJ...333..161L,1990ApJ...363...21C,1990MNRAS.247..166R,1994A&A...292...20R,1994ApJS...94...63B,1994ApJS...95..107W,1995ApJ...448..315W,1996A&A...315L..32C,1996AJ....112..839C,...,None,None,None,None,None,None,None,None,None,None
180782,1999GSLSP.168..255M,1971CaJES...8..523I,1973JPet...14..381L,1978E&PSL..37..380L,1978GeCoA..42.1199E,1979CoMP...69..105T,1980JSG.....2...63P,1981CoMP...76..177F,1982JGR....87.3849C,1983AmJS..283..993N,...,None,None,None,None,None,None,None,None,None,None
180783,2000fepc.conf..323C,1975A&A....40..303M,1981A&A...102...25B,1982A&A...107..283M,1983ARA&A..21..271I,1984A&A...136..355B,1985A&A...150...33B,1987ASSL..132..213C,1989A&A...219..167C,1990A&A...232...31X,...,None,None,None,None,None,None,None,None,None,None


## Helper functions

In [8]:
def filter_publications_by_authors(publications, authors_to_filter):
    """Filter publications by specified authors."""
    if authors_to_filter:
        pattern = '|'.join(authors_to_filter)
        return publications[publications['Author'].str.contains(pattern, case=False, na=False)]
    return publications

def create_year_author_maps(queried_publications):
    """Create mappings for years and authors from queried publications."""
    year_map = queried_publications.set_index('Bibcode')['Year'].to_dict()
    author_map = queried_publications.set_index('Bibcode')['Author'].to_dict()
    return year_map, author_map

## Direct citation node- and edgelist
We define a function to create the edgelist from the dataframe, which can be filtered for entries, where the citation count is above a certain threshold. It gives out a dataframe in a format, that after writing it to a SQL database, can be directly loaded into [gephi](https://gephi.org/).

In [9]:
def create_direct_citation_edgelist(df_all_bibcodes, queried_publications, min_cit_count=1, authors_to_filter=None):
    """Create direct citation edgelist from queried publications and references."""
    
    year_map, author_map = create_year_author_maps(queried_publications)  # Generate year/author maps
    data = []
    for _, row in df_all_bibcodes.iterrows():
        source = row['source']
        year = year_map.get(source)  # Lookup year for the source
        if year is None:  # Skip if year is missing
            continue
        # Filter publications by authors if specified
        if authors_to_filter and not any(author.lower() in author_map.get(source, "").lower() for author in authors_to_filter):
            continue
        # Collect non-null targets and create edge data
        targets = [target for target in row[1:] if target is not None]
        data.extend({'source': source, 'target': target, 'count': i, 'year': year} for i, target in enumerate(targets))
    df_edgelist = pd.DataFrame(data)
    # Filter out targets below the minimum citation count
    citation_counts = df_edgelist.groupby('target')['target'].transform('size')
    df_edgelist = df_edgelist[citation_counts >= min_cit_count]
    df_edgelist.insert(0, 'id', range(len(df_edgelist)))  # Add unique ID for each row
    return df_edgelist[['id', 'source', 'target', 'count', 'year']]

## Co-citation node- and edgelist

Same as for the direct citation network, we define a function to create the edgelist from the combined dataframe, which can be filtered for entries, where the co-citation count is above a certain threshold and again give it out as a SQL database and two CSV-Files.

In [10]:
def create_co_citation_edgelist(df_all_bibcodes, queried_publications, min_co_cit_count=1, authors_to_filter=None):
    """Create co-citation edgelist from queried publications and references."""
    
    queried_publications = filter_publications_by_authors(queried_publications, authors_to_filter)  # Filter by authors
    year_map, _ = create_year_author_maps(queried_publications)  # Generate year map
    co_citation_data = []
    for _, row in df_all_bibcodes.iterrows():
        source = row['source']
        year = year_map.get(source)  # Lookup year for the source
        if year is None:  # Skip if year is missing
            continue
        # Create co-citation pairs from non-null references
        cocit_refs = [ref for ref in row[1:] if ref is not None]
        co_citation_data.extend({'cocit_source': source, 'source': r1, 'target': r2, 'year': year} for r1, r2 in itertools.combinations(cocit_refs, 2))
    df_co_citation = pd.DataFrame(co_citation_data)
    # Count and filter pairs by minimum co-citation count
    df_co_citation['pair_count'] = df_co_citation.groupby(['source', 'target'])['cocit_source'].transform('count')
    df_filtered = df_co_citation[df_co_citation['pair_count'] >= min_co_cit_count]
    df_filtered.insert(0, 'id', range(len(df_filtered)))  # Add unique ID for each row
    return df_filtered[['id', 'year', 'source', 'target', 'cocit_source']]

## Bibliographical Coupling node- and edgelist

Same as twice before, we define a function to create the edgelist from the combined dataframe, which can be filtered for entries, where the bibliographic-coupling count is above a certain threshold and again give it out as a SQL database and two CSV-Files.

In [11]:
def create_bibliographic_coupling_edgelist(queried_publications, min_shared_refs=1, authors_to_filter=None):
    """Create a bibliographic coupling edgelist from queried publications."""

    queried_publications = filter_publications_by_authors(queried_publications, authors_to_filter)  # Filter by authors
    df_refs = queried_publications.explode('References').dropna(subset=['References'])  # Normalize references
    # Map each reference to a list of citing publications
    ref_source_map = df_refs.groupby('References')['Bibcode'].agg(list).to_dict()
    biblio_coupling_data = []
    for ref, sources in ref_source_map.items():
        # Create all pairs of sources sharing the same reference
        biblio_coupling_data.extend({'source': s1, 'target': s2, 'shared_ref': ref} for s1, s2 in itertools.combinations(sources, 2))
    df_edgelist = pd.DataFrame(biblio_coupling_data)
    year_map, _ = create_year_author_maps(queried_publications)  # Generate year map
    df_edgelist['year'] = df_edgelist['source'].map(year_map).astype('Int64')  # Map publication years
    # Filter pairs by minimum shared references
    df_edgelist['pair_count'] = df_edgelist.groupby(['source', 'target'])['shared_ref'].transform('count')
    df_filtered = df_edgelist[df_edgelist['pair_count'] >= min_shared_refs].drop_duplicates(subset=['source', 'target', 'shared_ref'])
    df_filtered.insert(0, 'id', range(len(df_filtered)))  # Add unique ID for each row
    return df_filtered[['id', 'year', 'source', 'target', 'shared_ref']]

## Author-Cocitation node- and edgelist


In [12]:
def create_author_co_citation_edgelist(filtered_publications, queried_references, min_co_cit_count=1, authors_to_filter=None):
    """Create a first author co-citation edgelist from filtered publications."""
    
    filtered_publications = filter_publications_by_authors(filtered_publications, authors_to_filter)  # Filter by authors
    # Extract first authors from queried references
    queried_references['FirstAuthor'] = queried_references['Author'].apply(lambda x: x.split(',')[0].strip() if pd.notnull(x) else None)
    ref_to_first_author = queried_references.set_index('Bibcode')['FirstAuthor'].to_dict()
    # Normalize references and map to first authors
    df_exp = filtered_publications.explode('References').dropna(subset=['References'])
    df_exp['first_author'] = df_exp['References'].map(ref_to_first_author).dropna()
    # Group publications by Bibcode and year to list first authors
    grouped = df_exp.groupby(['Bibcode', 'Year'])['first_author'].agg(list).reset_index()
    co_citation_data = []
    for _, row in grouped.iterrows():
        authors_set = sorted(set(row['first_author']))
        # Create all pairs of co-cited first authors
        co_citation_data.extend({'year': row['Year'], 'source': a1, 'target': a2, 'source_citation': row['Bibcode']} for a1, a2 in itertools.combinations(authors_set, 2))
    df_co = pd.DataFrame(co_citation_data)
    # Filter pairs by minimum co-citation count
    df_co['pair_count'] = df_co.groupby(['source', 'target'])['source_citation'].transform('count')
    df_filtered = df_co[df_co['pair_count'] >= min_co_cit_count]
    df_filtered.insert(0, 'id', range(len(df_filtered)))  # Add unique ID for each row
    return df_filtered[['id', 'year', 'source', 'target', 'source_citation']]

# Process Citations

In [13]:
def process_all_citations(df_all_bibcodes, queried_publications, queried_references, df_all_nodes,
                          output_type, metrics_to_calculate, min_counts, authors_to_filter=None):
    """Processes selected citation metrics."""

    suffix = "_filtered" if authors_to_filter else ""

    metric_functions = {
        'direct': create_direct_citation_edgelist,
        'co_citation': create_co_citation_edgelist,
        'bibliographic_coupling': create_bibliographic_coupling_edgelist,
        'author_co_citation': create_author_co_citation_edgelist
    }

    for metric in metrics_to_calculate:
        print(f"Processing {metric.capitalize()}...")
        min_count = min_counts.get(metric, 1)

        if metric in ['direct', 'co_citation']:
            df_edges = metric_functions[metric](df_all_bibcodes, queried_publications, min_count, authors_to_filter)
        elif metric == 'bibliographic_coupling':
            df_edges = metric_functions[metric](queried_publications, min_count, authors_to_filter)
        elif metric == 'author_co_citation':
            df_edges = metric_functions[metric](queried_publications, queried_references, min_count, authors_to_filter)
        else:
            raise ValueError(f"Invalid metric: {metric}")

        if not df_edges.empty:
            if metric == 'direct':
                node_columns = ['source', 'target']
            elif metric == 'co_citation':
                node_columns = ['source', 'target', 'cocit_source']
            elif metric == 'bibliographic_coupling':
                node_columns = ['source', 'target', 'shared_ref']
            elif metric == 'author_co_citation':
                node_columns = ['source', 'target', 'source_citation']
            else:
                raise ValueError(f"Invalid metric: {metric}")

            all_nodes = pd.concat([df_edges[col] for col in node_columns]).unique()
            df_filtered_nodes = df_all_nodes[df_all_nodes['id'].isin(all_nodes)]
            db_name = f'2_data/{metric}_database{suffix}.db'
            edge_table_cols = 'id text, year number, ' + ', '.join([f'{col} text' for col in node_columns])

            if output_type == 'db':
                process_db(db_name, edge_table_cols, df_edges, df_filtered_nodes)
            elif output_type == 'csv':
                csv_name = f'2_data/{metric.capitalize()}_CSV{suffix}'
                save_to_csv(df_edges, df_filtered_nodes, csv_name)
            else:
                raise ValueError("Invalid output type specified.")

            del df_edges, df_filtered_nodes
            gc.collect()
        else:
            print(f"No entries found for '{metric}'.")

In [14]:
def get_aggregated_data_from_db(authors_to_filter=None):
    """
    Query each metric's database to get aggregated counts by year.
    
    Returns:
        aggregated_data (dict): Metrics mapped to DataFrames [year, count_per_year].
    """
    suffix = "_filtered" if authors_to_filter else ""  # Add suffix if authors are filtered
    metric_dbs = {
        'direct': f'2_data/direct_database{suffix}.db',
        'co_citation': f'2_data/co_citation_database{suffix}.db',
        'bibliographic_coupling': f'2_data/bibliographic_coupling_database{suffix}.db',
        'author_co_citation': f'2_data/author_co_citation_database{suffix}.db'
    }

    aggregated_data = {}
    for metric, db_path in metric_dbs.items():
        try:
            conn = sqlite3.connect(db_path)  # Connect to the database
            df_counts = pd.read_sql_query(  # Query to aggregate counts per year
                "SELECT year, COUNT(*) as count_per_year FROM edges GROUP BY year;", conn
            )
            conn.close()
            if not df_counts.empty:
                aggregated_data[metric] = df_counts
        except Exception:  # Handle missing DB or empty results gracefully
            pass

    return aggregated_data

def plot_citation_metrics_from_aggregated(aggregated_data, start_year, end_year, slice_size=1, y_axis_type='log'):
    """
    Plots citation metrics from aggregated data over time slices.
    """
    plot_info = {  # Define names and colors for each metric
        'direct': {'name': 'Direct Citations', 'color': '#1f77b4'},
        'co_citation': {'name': 'Co-Citations', 'color': '#ff7f0e'},
        'author_co_citation': {'name': 'Author Co-Citations', 'color': '#2ca02c'},
        'bibliographic_coupling': {'name': 'Bibliographic Couplings', 'color': '#d62728'}
    }

    years = np.arange(start_year, end_year + 1, slice_size)  # Time slices
    citation_sums = {}
    total_counts = {}

    for metric, df_counts in aggregated_data.items():
        # Filter data to the desired year range and aggregate by time slice
        df_counts = df_counts[(df_counts['year'] >= start_year) & (df_counts['year'] <= end_year)].copy()
        df_counts['Slice'] = df_counts['year'] - (df_counts['year'] % slice_size)  # Group into slices
        citation_sums[metric] = df_counts.groupby('Slice')['count_per_year'].sum().to_dict()  # Summed counts
        total_counts[metric] = df_counts['count_per_year'].sum()  # Total count for the metric

    # Create subplots for each metric
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                        subplot_titles=[f"Total {plot_info[m]['name']}: {total_counts.get(m, 0)}"
                                        for m in plot_info],
                        vertical_spacing=0.1)

    for idx, metric in enumerate(plot_info, start=1):
        counts = citation_sums.get(metric, {})
        y_values = [counts.get(year, 0) for year in years]  # Values for each slice
        # Add a bar trace for the current metric
        fig.add_trace(go.Bar(x=years, y=y_values,
                             name=plot_info[metric]['name'],
                             marker_color=plot_info[metric]['color']),
                      row=idx, col=1)
        fig.update_yaxes(type=y_axis_type, title_text="Count", row=idx, col=1)  # Set y-axis type

    fig.update_xaxes(title_text="Year", row=4, col=1)  # Add x-axis title to the bottom plot
    fig.update_layout(height=1200, showlegend=False)  # Configure layout
    fig.show()

In [15]:
# Set parameters
start_year = 1911
end_year = 2000
slice_size = 1
y_axis_type = 'log'
authors_to_filter = ['Treder', 'Kreisel']  # or None
metrics_to_calculate = ['direct', 'co_citation', 'bibliographic_coupling', 'author_co_citation'] # ['author_co_citation']
min_counts = {'direct': 1, 'co_citation': 1, 'bibliographic_coupling': 1, 'author_co_citation': 1} # {'author_co_citation': 1}

process_all_citations(df_all_bibcodes, queried_publications, queried_references, df_all_nodes, 'db', metrics_to_calculate, min_counts, authors_to_filter)

Processing Direct...


C:\Users\rapha\anaconda3\envs\ADS_env\lib\site-packages\pandas\core\generic.py:2872: UserWarning:

The spaces in these column names will not be changed. In pandas versions < 0.14, spaces were converted to underscores.



Processing Co_citation...


Processing Bibliographic_coupling...


Processing Author_co_citation...


In [16]:
#| column: page
#| label: get_aggregated_data_from_db

# Get aggregated data from the newly created databases
aggregated_data = get_aggregated_data_from_db(authors_to_filter=authors_to_filter)

# Plot the citation metrics from aggregated data
plot_citation_metrics_from_aggregated(aggregated_data, start_year, end_year, slice_size, y_axis_type)

In [17]:
#| label: author co-citation database

# Load the database and display the first few rows of the edge table
db_name = '2_data/direct_database.db'
conn = sqlite3.connect(db_name)
df_edge_table = pd.read_sql_query("SELECT * FROM edges", conn)
df_edge_table.head(20)

,id,source,target,count,year
0,0,1914AnP...349..321E,1913AnP...347..533N,0,1914
1,1,1916AnP...355..955K,1911AnP...340..898E,0,1916
2,2,1916AnP...355..955K,1912AnP...343..355E,1,1916
3,3,1916AnP...355..955K,1914AnP...349..701K,2,1916
4,4,1916AnP...355..955K,1914AnP...350..481K,3,1916
5,5,1916AnP...355..955K,1916AnP...354..769E,4,1916
6,6,1916AnP...356..119G,1916AnP...354..769E,0,1916
7,7,1916AnP...356..639E,1916AnP...355..955K,0,1916
8,8,1916MNRAS..76..699D,1916SPAW.......688E,0,1916
9,9,1916skpa.conf..424S,1916AbhKP1916..189S,0,1916


## Web of Science Format Export
We can also export the data in the Web of Science or ``WOS`` format that can be imported into various secondary analysis tools like [Citespace](https://citespace.podia.com/) or [VOSviewer](https://www.vosviewer.com/).

In [18]:
# Convert JSON data to DataFrames
df_publications = pd.DataFrame(queried_publications)
df_references = pd.DataFrame(queried_references)

# Remove duplicates based on Bibcode
df_publications.drop_duplicates(subset='Bibcode', inplace=True)
df_references.drop_duplicates(subset='Bibcode', inplace=True)

# Create a lookup dictionary for references
ref_lookup = df_references.set_index('Bibcode').to_dict(orient='index')

def format_author(authors):
    # Format author names as "Last, First Middle"
    return '\n   '.join([f"{' '.join(author.split()[:-1])}, {author.split()[-1]}" for author in authors.split(', ')])

def format_references(ref_authors):
    # Extract and format the first author's name
    first_author = ref_authors.split(', ')[0]
    return f"{' '.join(first_author.split()[:-1])} {first_author.split()[-1]}"

def extract_affiliation(affiliation_str, author_index):
    # Extract affiliation based on index from the structured string
    aff_dict = {}
    for part in affiliation_str.split('; '):
        if '(' in part and ')' in part:
            key, value = part.split('(', 1)
            aff_dict[key.strip()] = value.strip(')')
    affiliation = aff_dict.get("AA", "N/A")
    return affiliation if affiliation != '-' else None

def format_publication(pub, ref_lookup):
    # Format a single publication in WoS format
    authors = format_author(pub['Author'])
    formatted_pub = (
        f"PT J\nAU {authors}\nTI {pub['Title_en']}\nSO {pub['Journal']}\nDT {pub['Category']}\n"
    )

    if 'Affiliation' in pub and pub['Affiliation']:
        affiliation = extract_affiliation(pub['Affiliation'], 1)
        if affiliation:
            formatted_pub += f"C1 {affiliation}\n"

    if 'Keywords' in pub and pub['Keywords']:
        formatted_pub += f"DE {pub['Keywords']}\n"

    if 'Abstract' in pub and pub['Abstract']:
        formatted_pub += f"AB {pub['Abstract_en']}\n"

    references = pub.get('References', [])
    formatted_refs = []
    for ref in references:
        ref_data = ref_lookup.get(ref, {})
        if ref_data:
            first_author = format_references(ref_data.get('Author', 'Unknown'))
            journal = ref_data.get('Journal', 'Unknown')
            volume = ref_data.get('Volume', '')
            start_page = ref_data.get('First Page', '')
            formatted_ref = f"{first_author}, {ref_data.get('Year', 'Unknown')}, {journal}"
            if volume:
                formatted_ref += f", V{volume}"
            if start_page:
                formatted_ref += f", P{start_page}"
            formatted_refs.append(formatted_ref)

    if formatted_refs:
        formatted_pub += "CR " + "\n   ".join(formatted_refs).replace('.', '') + "\n"

    formatted_pub += f"PY {pub['Year']}\n"

    for field in ['Volume', 'Issue', 'First Page', 'Last Page', 'DOI', 'Citation Count', 'Bibcode']:
        if field in pub and pub[field]:
            formatted_pub += f"{field[:2].upper()} {pub[field]}\n"

    if 'First Page' in pub and 'Last Page' in pub and pub['First Page'] and pub['Last Page']:
        try:
            first_page = int(pub['First Page'])
            last_page = int(pub['Last Page'])
            formatted_pub += f"PG {last_page - first_page + 1}\n"
        except ValueError:
            pass

    formatted_pub += "ER\n"
    return formatted_pub

# Generate formatted publications
formatted_publications = [format_publication(pub, ref_lookup) for pub in df_publications.to_dict(orient='records')]

# Set file name dynamically based on filtering
suffix = "_filtered" if authors_to_filter else ""
output_file_name = f'2_data/download_output_wos_format{suffix}.txt'

# Write to output file
with open(output_file_name, 'w', encoding='utf-8') as f:
    f.write("\n".join(formatted_publications))